# 🏗️ Diccionario de Datos: Proyecto AlquilerCaracas

Este cuaderno lee los datos desde una base de datos SQLite relacional que utiliza un modelo **Padre-Hijo (Histórico)**. Esto permite rastrear los cambios de precio de un mismo inmueble a lo largo del tiempo sin duplicar su información base.

## 1. Tabla Padre: `inmuebles`
Almacena la identidad única y estática de cada propiedad.

| Campo | Tipo SQLite | Descripción |
| :--- | :--- | :--- |
| **`id`** | `INTEGER` | Clave primaria generada por la base de datos. |
| **`source_name`** | `VARCHAR` | El portal de origen (ej. *rentahouse*, *remax*, *quarto*). |
| **`external_id`** | `VARCHAR` | El código único del inmueble en el portal de origen. |
| **`url`** | `VARCHAR` | El enlace directo a la publicación. |
| **`created_at`** | `DATETIME` | Fecha en la que descubrimos este inmueble por primera vez. |
| **`updated_at`** | `DATETIME` | Fecha de la última vez que revisamos este inmueble. |

> **Nota Arquitectónica:** La combinación de `source_name` + `external_id` es **ÚNICA**. El script `worker.py` usa esto para saber si un inmueble es nuevo o si ya existe en nuestra base de datos.

---

## 2. Tabla Hijo: `inmuebles_snapshots`
Almacena las "fotografías" dinámicas del inmueble en un momento dado.

| Campo | Tipo SQLite | ¿Cómo lo guardó el Worker? |
| :--- | :--- | :--- |
| **`id`** | `INTEGER` | Clave primaria del snapshot. |
| **`inmueble_id`** | `INTEGER` | Clave foránea que lo conecta con la tabla `inmuebles`. |
| **`precio`** | `FLOAT` | El precio extraído y limpio (solo números). |
| **`moneda`** | `VARCHAR` | Por defecto establecido en "USD". |
| **`titulo`** | `VARCHAR` | El título del anuncio. |
| **`descripcion`** | `TEXT` | La descripción completa limpia de HTML. |
| **`ubicacion`** | `JSON / TEXT` | Diccionario serializado con llaves: `{'municipio': '...', 'urbanismo': '...'}` |
| **`caracteristicas`** | `JSON / TEXT` | Diccionario serializado con llaves: `{'m2_totales': 100, 'habitaciones': 3, 'banos': 2}` |
| **`condiciones`** | `JSON / TEXT` | *(Opcional)* Diccionario para meses de adelanto, depósito, etc. |
| **`raw_extra_data`** | `JSON / TEXT` | *(Opcional)* Diccionario para amenidades (ej. Piscina, Pozo de agua). |
| **`scraped_at`** | `DATETIME` | La fecha y hora exacta en la que se tomó esta "fotografía" de los datos. |

---

## ⚙️ ¿Cómo funciona el Guardado Histórico en `worker.py`?
1. **Micro-Batching:** Los scrapers extraen los datos en lotes de 100 URLs.
2. **Empaquetado (Builder):** El `PropertySnapshotBuilder` agrupa atributos dispersos (como m2 y habitaciones) y los empaqueta en diccionarios de Python, que SQLite guarda como texto plano (JSON-like) en los campos `ubicacion` y `caracteristicas`.
3. **Decisión de Guardado (`procesar_y_guardar`):** * Si el inmueble **no existe**: Crea el Padre (`inmuebles`) y su primer Hijo (`inmuebles_snapshots`).
   * Si el inmueble **ya existe**: Compara el `precio` recién extraído con el `precio` del último snapshot guardado. **Solo guarda un nuevo snapshot si el precio cambió**, manteniendo la base de datos ligera y enfocada en eventos reales del mercado.

In [2]:
import sqlite3
import pandas as pd

# 1. Conectarnos a la base de datos
conn = sqlite3.connect('inmuebles.db')

# 2. Consulta ajustada al modelo real (pidiendo 'ubicacion' y 'caracteristicas')
query = """
SELECT
    i.source_name as portal,
    i.external_id,
    s.precio,
    s.moneda,
    s.titulo,
    s.ubicacion,
    s.caracteristicas,
    s.scraped_at as fecha_extraccion
FROM inmuebles i
JOIN inmuebles_snapshots s ON i.id = s.inmueble_id
WHERE s.id = (
    SELECT MAX(id)
    FROM inmuebles_snapshots
    WHERE inmueble_id = i.id
)
"""

# 3. Cargar los datos en Pandas
df = pd.read_sql_query(query, conn)

# 4. Mostrar información general
print("-" * 50)
print(f"✅ ¡ÉXITO TOTAL! Total de inmuebles listos para analizar: {len(df)}")
print("-" * 50)

# 5. Renderizar la tabla
display(df.head())

--------------------------------------------------
✅ ¡ÉXITO TOTAL! Total de inmuebles listos para analizar: 9820
--------------------------------------------------


,portal,external_id,precio,moneda,titulo,ubicacion,caracteristicas,fecha_extraccion
0,bolsainmobiliaria,6898540,500.0,USD,Alquilo apto 39m2 1h/1b/1p El Hatillo 8540,"{""municipio"": ""Caracas""}",{},2026-03-24 18:44:05.124181
1,bolsainmobiliaria,7251148,35000.0,USD,LOS JABILLOS - GUATIRE,"{""municipio"": ""Caracas"", ""urbanismo"": ""GUATIRE""}",{},2026-03-24 18:44:05.109229
2,bolsainmobiliaria,7292017,30000.0,USD,"Pent House en venta - Guatire, Calle Zamora","{""municipio"": ""Caracas"", ""urbanismo"": ""Guatire...","{""m2_totales"": 177.0, ""habitaciones"": 5, ""bano...",2026-03-24 18:44:05.107467
3,bolsainmobiliaria,7301988,600000.0,USD,"Apartamento en venta - Colinas de Bello Monte,...","{""municipio"": ""Caracas"", ""urbanismo"": ""Colinas...","{""m2_totales"": 408.0, ""habitaciones"": 3, ""bano...",2026-03-24 18:44:05.112269
4,bolsainmobiliaria,7302271,30000.0,USD,CARRETERA NACIONAL DE GUARENAS,"{""municipio"": ""Caracas""}",{},2026-03-24 18:44:05.108340


In [3]:
import ast

# 1. Función a prueba de balas para convertir el texto a un Diccionario real de Python
def texto_a_diccionario(texto):
    if pd.isna(texto) or texto == "":
        return {}
    try:
        # ast.literal_eval lee un string y lo convierte en un diccionario seguro
        return ast.literal_eval(str(texto))
    except (ValueError, SyntaxError):
        return {}

# 2. Convertimos las columnas de texto a diccionarios
df['ubicacion_dict'] = df['ubicacion'].apply(texto_a_diccionario)
df['carac_dict'] = df['caracteristicas'].apply(texto_a_diccionario)

# 3. ¡LA MAGIA! Expandimos los diccionarios para que cada llave sea una columna nueva
df_ubi = df['ubicacion_dict'].apply(pd.Series)
df_carac = df['carac_dict'].apply(pd.Series)

# 4. Unimos las columnas nuevas a nuestro DataFrame original
df_limpio = pd.concat([df, df_ubi, df_carac], axis=1)

# 5. Borramos las columnas viejas que ya no sirven para que quede ordenado
df_limpio = df_limpio.drop(columns=['ubicacion', 'caracteristicas', 'ubicacion_dict', 'carac_dict'])

# Mostramos el resultado de la transformación
print(f"Nuevas columnas generadas: {list(df_limpio.columns)}")
display(df_limpio.head())

Nuevas columnas generadas: ['portal', 'external_id', 'precio', 'moneda', 'titulo', 'fecha_extraccion', 'municipio', 'urbanismo', 'm2_totales', 'habitaciones', 'banos']


,portal,external_id,precio,moneda,titulo,fecha_extraccion,municipio,urbanismo,m2_totales,habitaciones,banos
0,bolsainmobiliaria,6898540,500.0,USD,Alquilo apto 39m2 1h/1b/1p El Hatillo 8540,2026-03-24 18:44:05.124181,Caracas,NaN,NaN,NaN,NaN
1,bolsainmobiliaria,7251148,35000.0,USD,LOS JABILLOS - GUATIRE,2026-03-24 18:44:05.109229,Caracas,GUATIRE,NaN,NaN,NaN
2,bolsainmobiliaria,7292017,30000.0,USD,"Pent House en venta - Guatire, Calle Zamora",2026-03-24 18:44:05.107467,Caracas,"Guatire, Calle Zamora",177.0,5.0,4.0
3,bolsainmobiliaria,7301988,600000.0,USD,"Apartamento en venta - Colinas de Bello Monte,...",2026-03-24 18:44:05.112269,Caracas,"Colinas de Bello Monte, Ohana Lofts",408.0,3.0,4.0
4,bolsainmobiliaria,7302271,30000.0,USD,CARRETERA NACIONAL DE GUARENAS,2026-03-24 18:44:05.108340,Caracas,NaN,NaN,NaN,NaN


In [4]:
import sqlite3
import pandas as pd

# 1. Nos conectamos y traemos las columnas que faltaban
conn = sqlite3.connect('inmuebles.db')
query_auditoria = """
SELECT
    i.source_name as portal,
    s.condiciones,
    s.raw_extra_data
FROM inmuebles i
JOIN inmuebles_snapshots s ON i.id = s.inmueble_id
WHERE s.id = (
    SELECT MAX(id)
    FROM inmuebles_snapshots
    WHERE inmueble_id = i.id
)
"""
df_auditoria = pd.read_sql_query(query_auditoria, conn)

# 2. Función para auditar la calidad de una columna JSON/Texto
def auditar_columna(df, nombre_columna):
    total = len(df)

    # Contamos nulos reales (NULL en base de datos)
    nulos = df[nombre_columna].isna().sum()

    # Contamos "falsos vacíos" (diccionarios vacíos '{}' o strings vacíos '')
    falsos_vacios = df[nombre_columna].astype(str).str.strip().isin(['', '{}', 'None', 'nan', '[]']).sum()

    # Lo que queda son datos reales
    con_datos = total - nulos - falsos_vacios

    print(f"📊 Auditoría de la columna: '{nombre_columna}'")
    print(f"  - 🟢 Con datos reales: {con_datos} ({(con_datos/total)*100:.1f}%)")
    print(f"  - 🔴 Nulos absolutos: {nulos}")
    print(f"  - 🟡 Diccionarios/Listas vacías ('{{}}', '[]'): {falsos_vacios}")
    print("-" * 40)

    # Si hay datos reales, mostramos 3 ejemplos al azar para ver qué capturaron los scrapers
    if con_datos > 0:
        ejemplos = df[~df[nombre_columna].isna() & ~df[nombre_columna].astype(str).str.strip().isin(['', '{}', 'None', 'nan', '[]'])]
        print("💡 Ejemplos de lo que hay adentro:")
        display(ejemplos[['portal', nombre_columna]].sample(min(3, con_datos)))
    print("\n")

# 3. Ejecutamos la auditoría
auditar_columna(df_auditoria, 'condiciones')
auditar_columna(df_auditoria, 'raw_extra_data')

📊 Auditoría de la columna: 'condiciones'
  - 🟢 Con datos reales: 0 (0.0%)
  - 🔴 Nulos absolutos: 0
  - 🟡 Diccionarios/Listas vacías ('{}', '[]'): 9820
----------------------------------------


📊 Auditoría de la columna: 'raw_extra_data'
  - 🟢 Con datos reales: 7610 (77.5%)
  - 🔴 Nulos absolutos: 0
  - 🟡 Diccionarios/Listas vacías ('{}', '[]'): 2210
----------------------------------------
💡 Ejemplos de lo que hay adentro:


,portal,raw_extra_data
6409,rentahouse,"{""amenidades"": [""Ceramica"", ""Natural"", ""110Vlt..."
453,mlscaracas,"{""amenidades"": [""Ba\u00f1o auxiliar"", ""Ba\u00f..."
630,mlscaracas,"{""amenidades"": [""Ba\u00f1o auxiliar"", ""Ba\u00f..."
